---
title: "数论——质数算法"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

## 📝 题目 204：计数质数

::: {.callout-caution}
### 📖 题目描述
给定整数 `n` ，返回 **所有小于非负整数 `n` 的质数的数量** 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 10`
    * **输出**：`4`
    * **解释**：小于 10 的质数一共有 4 个, 它们是 2, 3, 5, 7 。

* **示例 2**：
    * **输入**：`n = 0`
    * **输出**：`0`

* **示例 3**：
    * **输入**：`n = 5000000`
    * **输出**：`348513`

**⚠️ 极限数据警告**：
`0 <= n <= 5 * 10^6` （500 万！暴力法绝对会 Time Limit Exceeded）
:::

In [2]:
class Solution204:
    def countPrimes(self, n: int) -> int:
        if n < 3:
            return 0
        prime = [True] * n
        prime[0], prime[1] = False, False  # 0 和 1 明确不是质数
        for i in range(2, int(n ** 0.5) + 1):  # 只需扫描到 根号n 即可，大幅减少 Python 循环开销
            if prime[i]:
                length = (n - 1 - i * i) // i + 1  # 计算这里面一共有多少个元素
                prime[i * i : n : i] = [False] * length  # 切片批量赋值
        return sum(prime)

In [3]:
#| code-fold: true
def test_204(func):
    test_cases = [
        {"desc": "示例 1: 常规数字 10", "n": 10, "expected": 4}, # 2, 3, 5, 7
        {"desc": "示例 2: 边界 0", "n": 0, "expected": 0},
        {"desc": "示例 3: 边界 1", "n": 1, "expected": 0},
        {"desc": "边界: 刚好等于质数 2 (求小于2的)", "n": 2, "expected": 0},
        {"desc": "边界: 刚好等于质数 3 (求小于3的)", "n": 3, "expected": 1}, # 只有 2
        {"desc": "常规中等数字 100", "n": 100, "expected": 25},
        {"desc": "偏大数字 10000", "n": 10000, "expected": 1229},
        {"desc": "极限压测 1: 100万", "n": 1000000, "expected": 78498},
        {"desc": "满级 Boss: 500万 (LeetCode 极限用例)", "n": 5000000, "expected": 348513}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<8} | {'实际':<8} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["n"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<3} | {status:<6} | {tc['expected']:<8} | {actual:<8} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_204(Solution204().countPrimes)

ID  | 结果     | 预期       | 实际       | 描述
-----------------------------------------------------------------
1   | ✅ PASS | 4        | 4        | 示例 1: 常规数字 10
2   | ✅ PASS | 0        | 0        | 示例 2: 边界 0
3   | ✅ PASS | 0        | 0        | 示例 3: 边界 1
4   | ✅ PASS | 0        | 0        | 边界: 刚好等于质数 2 (求小于2的)
5   | ✅ PASS | 1        | 1        | 边界: 刚好等于质数 3 (求小于3的)
6   | ✅ PASS | 25       | 25       | 常规中等数字 100
7   | ✅ PASS | 1229     | 1229     | 偏大数字 10000
8   | ✅ PASS | 78498    | 78498    | 极限压测 1: 100万
9   | ✅ PASS | 348513   | 348513   | 满级 Boss: 500万 (LeetCode 极限用例)
-----------------------------------------------------------------
测试总结: 通过 9/9


::: {.callout-important}
### 💡 思路讲解

古希腊数学家埃拉托斯特尼发明了一种极其聪明的物理方法：
不要去“找”质数，而是去**“划掉”所有不是质数的数（合数）**，剩下的自然全是质数！

**核心法则**：一个质数的各个倍数，绝对不是质数。

**算法流程**：

1. **铺开白纸**：创建一个长度为 `n` 的布尔数组 `is_prime`，一开始假设所有人都是质数（全部填 `True`）。然后明确 `0` 和 `1` 不是质数（填 `False`）。
2. **从小到大扫描**：用一个变量 `i` 从 `2` 开始扫。
3. **发现质数，立刻划掉它的倍数**：
   如果 `is_prime[i]` 是 `True`（说明我们遇到了一个真正的质数）：
   - 我们就从 `i * i` 开始，每次跨步 `i`，把它的倍数 `i*i, i*i+i, i*i+2i ...` 全局划掉（设为 `False`）。
   *(为什么从 `i * i` 开始？因为比它小的倍数比如 `2 * i`、`3 * i`，早就被前面扫过的质数 2 和 3 给划掉啦！)*
4. **提前下班优化**：外层扫描质数的循环，其实只需要扫到 $\sqrt{n}$ 就可以了！因为如果一个数有大于 $\sqrt{n}$ 的因数，它必然也有一个小于 $\sqrt{n}$ 的因数，早就被划掉了。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N \log \log N)$。这是一个极其贴近 $O(N)$ 的恐怖复杂度。500万的数据量在它面前就是一瞬间的事。
* **空间复杂度**：$O(N)$。我们需要一个长度为 `n` 的布尔数组来当“筛子”。
:::

## 📝 题目 365：水壶问题

::: {.callout-caution}
### 📖 题目描述
有两个水壶，容量分别为 `x` 和 `y` 升。另有一个无穷多的水资源。
你需要判断能否通过这两个水壶，准确地量出 `target` 升的水。

你可以进行以下三种操作：
1. 装满任意一个水壶。
2. 清空任意一个水壶。
3. 把水从一个水壶倒进另一个水壶，直到装满或者倒空。

**输入输出示例**：

* **示例 1**：
    * **输入**：`x = 3`, `y = 5`, `target = 4`
    * **输出**：`True`
    * **解释**：
      1. 装满 5 升壶（壶状态：0, 5）。
      2. 把 5 升壶的水倒进 3 升壶，装满（壶状态：3, 2）。
      3. 清空 3 升壶（壶状态：0, 2）。
      4. 把 5 升壶里剩下的 2 升水倒进 3 升壶（壶状态：2, 0）。
      5. 再次装满 5 升壶（壶状态：2, 5）。
      6. 把 5 升壶的水倒进 3 升壶直到装满。因为 3 升壶里已经有 2 升，所以只能倒进去 1 升。此时 5 升壶里剩下了 4 升！（成功量出 target）。

* **示例 2**：
    * **输入**：`x = 2`, `y = 6`, `target = 5`
    * **输出**：`False`
    * **解释**：2 和 6 怎么折腾，量出来的水都是偶数，绝不可能量出 5 升这种奇数。
:::

In [8]:
class Solution365:
    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        if x + y < target:  # 物理拦截：两个水壶加起来都装不下，绝对不可能
            return False
        if target == 0:  # 边界拦截：想要 0 升水，什么都不干就做到了
            return True
        def GCD(a: int, b: int) -> int:
            return a if b == 0 else GCD(b, a % b)
        return target % GCD(x, y) == 0  # 只要 target 是最大公约数的倍数，方程 ax + by = target 必然有解

In [9]:
#| code-fold: true
def test_365(func):
    test_cases = [
        {"desc": "示例 1: 经典的 3 和 5 量出 4", "x": 3, "y": 5, "target": 4, "expected": True},
        {"desc": "示例 2: 全是偶数，量不出奇数", "x": 2, "y": 6, "target": 5, "expected": False},
        {"desc": "示例 3: 完美拼凑", "x": 1, "y": 2, "target": 3, "expected": True},
        {"desc": "物理拦截: 目标水比两壶之和还大", "x": 1, "y": 1, "target": 12, "expected": False},
        {"desc": "边界: 只要 0 升", "x": 3, "y": 5, "target": 0, "expected": True},
        {"desc": "有一个壶是 0", "x": 0, "y": 2, "target": 1, "expected": False},
        {"desc": "有一个壶是 0 但刚好命中", "x": 0, "y": 2, "target": 2, "expected": True},
        {"desc": "大质数互质测试", "x": 104579, "y": 104593, "target": 12444, "expected": True}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<8} | {'实际':<8} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["x"], tc["y"], tc["target"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<3} | {status:<6} | {tc['expected']:<8} | {actual:<8} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_365(Solution365().canMeasureWater)

ID  | 结果     | 预期       | 实际       | 描述
-----------------------------------------------------------------
1   | ✅ PASS | 1        | 1        | 示例 1: 经典的 3 和 5 量出 4
2   | ✅ PASS | 0        | 0        | 示例 2: 全是偶数，量不出奇数
3   | ✅ PASS | 1        | 1        | 示例 3: 完美拼凑
4   | ✅ PASS | 0        | 0        | 物理拦截: 目标水比两壶之和还大
5   | ✅ PASS | 1        | 1        | 边界: 只要 0 升
6   | ✅ PASS | 0        | 0        | 有一个壶是 0
7   | ✅ PASS | 1        | 1        | 有一个壶是 0 但刚好命中
8   | ✅ PASS | 1        | 1        | 大质数互质测试
-----------------------------------------------------------------
测试总结: 通过 8/8


::: {.callout-important}
### 💡 思路讲解

**数学原理解析：**
不管你执行多少次装水、倒水、清空的操作，你水壶里水的总量，实际上就是在执行：
`水总量 = a * x + b * y` （其中 `a` 和 `b` 是整数，代表水壶被装满或清空的次数）。

这就引出了数论中大名鼎鼎的**裴蜀定理**：
**对于任意两个整数 $x$ 和 $y$，如果它们的最大公约数是 $d$，那么方程 $ax + by = z$ 有整数解的充要条件是：$z$ 必须是 $d$ 的倍数！**

也就是说，只要 `target` 是 `GCD(x, y)` 的倍数，我们在数学上就绝对能倒出这么多水！

**所以代码只需做两次拦截：**
1. **物理拦截**：如果 `x + y < target`，两个水壶加起来都装不下目标水量，直接返回 `False`。
2. **数论拦截**：如果 `target % GCD(x, y) == 0`，则返回 `True`，否则返回 `False`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(\log(\min(x, y)))$。这取决于辗转相除法求 GCD 的时间复杂度。对于普通的整数，这个时间几乎可以忽略不计。
* **空间复杂度**：$O(1)$。只需要常数级别的空间。
:::